# Naive Bayes from Scratch

The aim of this notebook is to implement the **Naive Bayes Classfier** algorithm from scratch using only NumPy.

## 1. Imports

In [30]:
# Imports
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB

## 2. Load dataset

In [2]:
# Load the dataset
URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(URL, sep='\t', header=None, names=['label', 'text'])

In [3]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print("Shape:\n", df.shape)
print("Head:\n", df.head())
print("Description:\n", df.describe())
print("Null Values:\n", df.isnull().sum())

Shape:
 (5572, 2)
Head:
   label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
Description:
        label                    text
count   5572                    5572
unique     2                    5169
top      ham  Sorry, I'll call later
freq    4825                      30
Null Values:
 label    0
text     0
dtype: int64


Dataset with: 5,572 SMS messages, 2 categories, no null values. The distribution is **skewed** (4,825 ham vs. ~747 spam).

## 3. Preprocessing

In [5]:
# Encoding labels into 0/1, ham=0, spam=1
df["label"] = df["label"].apply(lambda x: 0 if x == "ham" else 1)

In [6]:
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
# Prepare the data: target and features
X = df["text"]
y = df["label"].values

In [8]:
# Split datasets into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# CountVectorizer: convert texts into a matrix of token counts
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(X_train)  # build vocabulary + fit
X_test_counts = vectorizer.transform(X_test) # apply existing vocabulary

In [10]:
X_train_counts.shape, len(vectorizer.vocabulary_)

((4457, 7702), 7702)

**Notes**:

- **vectorizer.vocabulary_** is a dictionary mapping terms to feature indices (word -> column index).

- **X_train_counts** is a sparse matrix of shape (num_samples, num_features) containing the **token counts** for each sms/text in the training set. The number of features corresponds to the size of the vocabulary built by CountVectorizer.

The complete matrix:

- 4,457 rows → one SMS per row
- 7,702 columns → one vocabulary word per column
- value [i, j] → number of occurrences of word j in SMS i

And this matrix is **sparse** because the vast majority of values are 0, because each SMS uses only a tiny fraction of the vocabulary. 

In [11]:
# A mapping of termes to feature indices, word -> column index
# vectorizer.vocabulary_

## 4. From scratch implementation

In [12]:
# Training loop
def fit(X, y):
    """
    Fit the Naive Bayes model to the training data X and labels y.
    Returns the class priors and feature log probabilities.
    """
    # Store all unique classes 
    classes = np.unique(y)

    # Compute class priors log(P(y=c) for each class (see theory.md, section 7)
    class_priors = []
    for c in classes:
        class_priors.append(np.log(len(y[y == c]) / len(y)))
        
    # Store log(P(xi|y=c)) for each feature i and class c in a dictionary
    feature_log_probs = []

    # For each class, compute log(P(xi, y=c)) (see theory.md, section 9)
    for c in classes:
        # Extract rows of a sparse matrix corresponding to class c, (shape N_c, V) with Nc lines of class c and V features (vocabulary size)
        class_c = X[y == c]
        # Compute total number of occurences of the word i in all sms of class c
        n_i_c = np.asarray(class_c.sum(axis=0)).flatten() # to return a vector (V,) and not (1, V)
        # Compute P(xi|y=c) 
        p_xi_c = (n_i_c + 1) / (n_i_c.sum() + X.shape[1]) # V=X.shape[1], number of features X
        # Store log(P(xi|y=c)) for each feature i and class c in a dictionary
        feature_log_probs.append(np.log(p_xi_c))

    return np.array(class_priors), np.array(feature_log_probs)

In [13]:
# Test the training loop on the training data
class_priors, feature_log_probs = fit(X_train_counts, y_train)
print(class_priors.shape)
print(feature_log_probs.shape)

(2,)
(2, 7702)


- **class_priors**: (2,), correct with 2 values, one per class.

- **feature_log_probs**: (2, 7702), a vector of log-probabilities per class, one value per vocabulary word.

In [26]:
# Prediction pipeline (see theory.md, section 10)
def predict(X, class_priors, feature_log_probs):
    """
    Predict the class labels for the input data X using the fitted model.
    Returns the predicted class labels.
    """
    # For each class c, compute the score : score(c) = log(P(y=c)) + sum of i of (xi * log(P(xi, y=c)))
    scores = X @ feature_log_probs.T
    scores = scores + class_priors

    return np.argmax(scores, axis=1)

In [27]:
y_pred = predict(X_test_counts, class_priors, feature_log_probs)
print(y_pred.shape)
print(y_pred[:10])

(1115,)
[0 0 0 0 0 0 0 0 0 0]


Looking good. The top 10 results are all “ham”, which makes sense given the dataset's imbalance (86% ham).

Also, **1,115** is exactly the **number of SMS messages in the test set** (20% of 5,572). So one prediction per SMS, that's what we'd expect.

In [29]:
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

0.9919282511210762
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       966
           1       1.00      0.94      0.97       149

    accuracy                           0.99      1115
   macro avg       1.00      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



99.2% accuracy, it's an excellent result for a scratch implementation.

A few observations on the classification report:

- **Spam precision (1) = 1.00**, when the model predicts spam, it is always correct.
- **Spam recall (1) = 0.94**, it **misses ~6% of spam** (false negatives).

This is an acceptable trade-off: it’s better to let a few spam emails through than to block legitimate emails.

In [32]:
# Comparison with Scikit-Learn's Multinomial Naive Bayes
clf = MultinomialNB()
clf.fit(X_train_counts, y_train)
y_pred_sklearn = clf.predict(X_test_counts)
print(accuracy_score(y_test, y_pred_sklearn))
print(classification_report(y_test, y_pred_sklearn))

0.9919282511210762
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       966
           1       1.00      0.94      0.97       149

    accuracy                           0.99      1115
   macro avg       1.00      0.97      0.98      1115
weighted avg       0.99      0.99      0.99      1115



Identical results, implementation validated.